<div dir="rtl">

# الترم ٣ · الوحدة ١: معالجة اللغة (NLP)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdelrahman-abozarifa04/ai-for-kids-course/blob/main/notebooks/teens/term-3/08-nlp-sentiment.ipynb)

الكمبيوتر ما بيفهمش كلام، بيفهم **أرقام**. النهارده هنحوّل الجمل لأرقام، وندرّب موديل يعرف الجملة **إيجابية ولا سلبية** (Sentiment Analysis)، بالإنجليزي **وبالعامية المصرية**.

</div>

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

<div dir="rtl">

## ١. البيانات: تعليقات على مطاعم ومحلات

`1` = إيجابي، `0` = سلبي. ٤٠ جملة بس، مكتوبة بإيدينا علشان نفهم. الموديلات الحقيقية بتتدرب على ملايين الجمل.

</div>

In [ ]:
POSITIVE = """
the food was amazing and the staff were friendly
great service and fast delivery
i loved the pizza it was delicious
excellent place highly recommended
very good prices and nice atmosphere
the koshari was perfect and hot
best burger i ever had
friendly people and clean tables
fast and tasty i will come again
wonderful experience thank you
the staff were kind and helpful
delicious food and great prices
amazing desserts i loved them
clean place and fast service
the pizza was hot and tasty
great experience i loved it
nice staff and delicious food
the delivery was fast and the food was great
excellent koshari and friendly staff
good food good people good prices
""".strip().split("\n")

NEGATIVE = """
the food was cold and tasteless
terrible service we waited an hour
i hated the pizza it was burnt
very expensive and the portions are small
dirty tables and rude staff
the delivery was late and the order was wrong
worst burger ever never again
bad experience i will not come back
slow and disappointing
the place was noisy and the food was bad
rude staff and terrible food
the pizza was cold and burnt
expensive and disappointing
dirty place and slow service
the order was wrong and late
bad food and bad service
i hated the koshari it was cold
terrible experience never again
the staff were rude and slow
worst delivery ever the food was cold
""".strip().split("\n")

reviews = [(t, 1) for t in POSITIVE] + [(t, 0) for t in NEGATIVE]
df = pd.DataFrame(reviews, columns=["text", "label"])
df.sample(5, random_state=0)

<div dir="rtl">

## ٢. Bag of Words: نعد الكلمات

كل جملة بتتحول لقائمة أرقام: كل كلمة في القاموس ليها خانة، والرقم = الكلمة اتكررت كام مرة.

</div>

In [ ]:
toy = ["good food", "bad food", "good good service"]
vec = CountVectorizer()
matrix = vec.fit_transform(toy)
pd.DataFrame(matrix.toarray(), columns=vec.get_feature_names_out(), index=toy)

<div dir="rtl">

## ٣. ندرّب الموديل

`make_pipeline` بيربط خطوتين: (١) نحوّل النص لأرقام بـ **TF-IDF** (زي عدّ الكلمات بس بيقلل أهمية الكلمات المنتشرة زي "the")، و(٢) نصنّف بـ **Logistic Regression**.

</div>

In [ ]:
model = make_pipeline(TfidfVectorizer(), LogisticRegression(C=10))
scores = cross_val_score(model, df["text"], df["label"], cv=5)
print(f"Cross-validated accuracy: {scores.mean():.0%}")
model.fit(df["text"], df["label"])

In [ ]:
def explain(model, top=8):
    vec, clf = model.named_steps["tfidfvectorizer"], model.named_steps["logisticregression"]
    words = vec.get_feature_names_out()
    order = np.argsort(clf.coef_[0])
    print("Most NEGATIVE words:", list(words[order[:top]]))
    print("Most POSITIVE words:", list(words[order[-top:]][::-1]))

explain(model)

<div dir="rtl">

الموديل **اتعلم لوحده** إن كلمات زي `great` و `friendly` إيجابية، و `bad` و `worst` سلبية. محدش كتبله قاموس!

</div>

In [ ]:
tests = ["the staff were amazing", "cold pizza and rude people", "the service was not good", "the food was not bad"]
for text, prob in zip(tests, model.predict_proba(tests)[:, 1]):
    print(f"{prob:.0%} positive  <-  {text}")

<div dir="rtl">

**لاحظ آخر جملتين:** في "not good" الموديل متلخبط (قريب من ٥٠٪) لأنه شايف كلمة `good` الإيجابية، وفي "not bad" قال إنها **سلبية** لأنه شايف `bad`. ده حد من حدود Bag of Words: **بيعد الكلمات ومش بيفهم ترتيبها**. الموديلات الحديثة (Transformers) بتحل المشكلة دي لأنها بتقرا الجملة كلها مع بعض.

</div>

<div dir="rtl">

## ٤. العامية المصرية 🇪🇬

العربي فيه تحديات زيادة: **أ/إ/آ/ا**، و**ة/ه**، و**ى/ي**، والتطويل (جمييييل). هنعمل دالة تنضيف (normalization).

</div>

In [ ]:
def normalize_arabic(text: str) -> str:
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[ً-ْ]", "", text)      # نشيل التشكيل
    text = re.sub(r"(.)\1{2,}", r"\1", text)         # جمييييل -> جميل
    return text

print(normalize_arabic("الأكل جمييييييل جداً والخدمة ممتازة"))

In [ ]:
egy = [
    ("الاكل حلو جدا والناس محترمة", 1),
    ("خدمة ممتازة والتوصيل سريع", 1),
    ("الكشري تحفة وسخن", 1),
    ("المكان نضيف والاسعار حلوة", 1),
    ("تجربة جميلة وهرجع تاني", 1),
    ("البيتزا جامدة جدا", 1),
    ("الاكل وحش وبارد", 0),
    ("الخدمة زفت واستنينا ساعة", 0),
    ("غالي اوي والكمية صغيرة", 0),
    ("المكان وسخ والعمال قلالات الذوق", 0),
    ("الاوردر اتأخر وجه غلط", 0),
    ("تجربة وحشة مش هرجع تاني", 0),
    ("العمال ذوق جدا والمكان هادي", 1),
    ("الحلويات تجنن والاسعار كويسة", 1),
    ("اكل نضيف وطعمه حلو", 1),
    ("التوصيل سريع والاكل سخن", 1),
    ("الاكل ملوش طعم", 0),
    ("التوصيل بطيء جدا", 0),
    ("المكان زحمة ودوشة", 0),
    ("الاسعار نار والاكل مش حلو", 0),
]
egy_df = pd.DataFrame(egy, columns=["text", "label"])
egy_model = make_pipeline(TfidfVectorizer(preprocessor=normalize_arabic, analyzer="char_wb", ngram_range=(2, 4)),
                          LogisticRegression(C=10))
egy_model.fit(egy_df["text"], egy_df["label"])

for text in ["الاكل تحفه والخدمه سريعه", "الاكل باااارد ووحش", "الاسعار غاليه اوي"]:
    p = egy_model.predict_proba([text])[0, 1]
    print(f"{p:.0%} positive  <-  {text}")

<div dir="rtl">

**ليه `analyzer="char_wb"`؟** بدل ما نقسم لكلمات، بنقسم لقطع حروف صغيرة (n-grams). ده بيساعد مع العامية لأن نفس الكلمة بتتكتب بطرق كتير، و"تحفه" و"تحفة" هيبقوا قريبين من بعض.

</div>

<div dir="rtl">

## 🏋️ تمرين ١
ضيف ٥ جمل إيجابية و٥ سلبية بالفرانكو (`el akl gamed`) للـ dataset المصري، وأعد التدريب وجرّب.

<details><summary>الحل</summary>

```python
franco = [("el akl gamed gedan", 1), ("el service momtaz", 1), ("7elw awy w rakhis", 1),
          ("tagroba 7elwa", 1), ("el makan nedif", 1),
          ("el akl we7esh", 0), ("ghaly awy", 0), ("el order etakhar", 0),
          ("el service zeft", 0), ("mesh harga3 tany", 0)]
all_df = pd.concat([egy_df, pd.DataFrame(franco, columns=["text", "label"])], ignore_index=True)
egy_model.fit(all_df["text"], all_df["label"])
print(egy_model.predict_proba(["el akl 7elw", "el akl ghaly w we7esh"])[:, 1].round(2))
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🏋️ تمرين ٢: نصلح "not good"
جرّب تخلي الموديل الإنجليزي يفهم النفي: حوّل `not X` لكلمة واحدة `not_X` قبل التدريب.

<details><summary>الحل</summary>

```python
def mark_negation(text):
    return re.sub(r"\bnot (\w+)", r"not_\1", text.lower())

neg_model = make_pipeline(TfidfVectorizer(preprocessor=mark_negation), LogisticRegression())
extra = pd.DataFrame([("the food was not good", 0), ("the service was not bad", 1),
                      ("not tasty at all", 0), ("not expensive and good", 1)], columns=["text", "label"])
neg_model.fit(pd.concat([df, extra])["text"], pd.concat([df, extra])["label"])
print(neg_model.predict_proba(["the pizza was not good"])[0, 1].round(2))
```
</details>

</div>

In [ ]:
# اكتب الحل هنا

<div dir="rtl">

## 🔭 الخطوة الجاية: Hugging Face

في Colab تقدر تجرّب موديل حقيقي متدرب على ملايين الجمل في ٣ سطور:

```python
!pip install -q transformers
from transformers import pipeline
clf = pipeline("sentiment-analysis")
clf("The service was not good")
```

## ✅ لخّص
- النص لازم يتحول لأرقام: **Bag of Words** / **TF-IDF**.
- **Sentiment analysis** = تصنيف النص (إيجابي/سلبي).
- العربي محتاج **normalization** وبيستفيد من **character n-grams**.
- Bag of Words ما بيفهمش الترتيب ← الحل: Transformers.

</div>